##### Create features from StreamSource

In [0]:
%pip install databricks-feature-engineering>=0.16.0
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering.entities import (
    Feature, DeltaTableSource, StreamSource, RequestSource, FieldDefinition,
    ScalarDataType, ColumnSelection, AggregationFunction,
    Count, LastDistinct, SlidingWindow, RollingWindow,
)


In [0]:
LastDistinct

In [0]:
from datetime import timedelta
from databricks.feature_engineering.entities import (
    Feature, StreamSource, AggregationFunction, Count, RollingWindow,
)

# Three-part Unity Catalog name of the Stream (created via fe.create_stream)
STREAM_SOURCE = "my_catalog.my_schema.my_clickstream"

# Specify a data source (StreamSource or DeltaTableSource)
session_source = StreamSource(
   full_name=STREAM_SOURCE,
)
# Specify an aggregation type
SESSION_WINDOW = RollingWindow(
   window_duration=timedelta(hours=24),
)
# Configure the feature computation
session_event_count_features = Feature(
   source=session_source,
   entity=["value.user_session"],
   timeseries_column="value.event_time",
   function=AggregationFunction(Count(input="value.event_type"), SESSION_WINDOW),
   name="session_event_count",
)


In [0]:
from datetime import timedelta
import mlflow
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_engineering.entities import (
    Feature, DeltaTableSource, StreamSource, RequestSource, FieldDefinition,
    ScalarDataType, ColumnSelection, AggregationFunction,
    Count, LastDistinct, SlidingWindow, RollingWindow,
)

fe = FeatureEngineeringClient()

# Batch: how popular is the product on the page? (7-day views, refreshed hourly)
clickstream = DeltaTableSource(
    catalog_name="retail", schema_name="recs", table_name="clickstream"
)
product_views_7d = Feature(
    name="product_views_7d",
    source=clickstream,
    entity=["product_id"],
    timeseries_column="event_time",
    function=AggregationFunction(
        operator=Count(input="event_id"),
        time_window=SlidingWindow(
            window_duration=timedelta(days=7), slide_duration=timedelta(hours=1)
        ),
    ),
)

# Streaming: what has this shopper looked at in the last 30 minutes?
session_events = StreamSource(full_name="retail.recs.session_events")
session_recent_products = Feature(
    name="session_recent_products",
    source=session_events,
    entity=["value.session_id"],
    timeseries_column="value.event_time",
    function=AggregationFunction(
        operator=LastDistinct(input="value.product_id", n=10),
        time_window=RollingWindow(window_duration=timedelta(minutes=30)),
    ),
)

# Request-time: context that only exists on this page view
page_context = RequestSource(schema=[
    FieldDefinition(name="search_query", data_type=ScalarDataType.STRING),
    FieldDefinition(name="device_type", data_type=ScalarDataType.STRING),
])
search_query = Feature(name="search_query", source=page_context,
                       function=ColumnSelection("search_query"))
device_type = Feature(name="device_type", source=page_context,
                      function=ColumnSelection("device_type"))

features = [product_views_7d, session_recent_products, search_query, device_type]

# One call joins all three cadences, point-in-time, for every labeled interaction
training_set = fe.create_training_set(
    df=labeled_interactions,  # session_id, product_id, event_time, search_query, device_type, clicked
    features=features,
    label="clicked",
)
model = train_recommender(training_set.load_df())

# The feature list travels with the model. To score an observation, only session_id, product_id, search_query, and device type are required as inputs. 
fe.log_model(
    model=model,
    artifact_path="recommender",
    flavor=mlflow.pyfunc,
    training_set=training_set,
    registered_model_name="retail.recs.session_recommender",
)


In [0]:
session_recent_products = Feature(
    name="session_recent_products",
    source=session_events,
    entity=["session_id"],
    timeseries_column="event_time",
    function=AggregationFunction(
        operator=LastDistinct(input="product_id", n=10),
        time_window=RollingWindow(window_duration=timedelta(minutes=30)),
    ),

In [0]:
from datetime import timedelta
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_engineering.entities import (
    Feature, StreamSource, AggregationFunction, LastDistinct, RollingWindow,
)

fe = FeatureEngineeringClient()

# Real StreamSource from the project
import yaml, os
with open(os.path.join(os.getcwd(), "config.yml")) as f:
    config = yaml.safe_load(f)
CATALOG = config["catalog"]
SCHEMA = config["schema"]
STREAM_SOURCE = f"{CATALOG}.{SCHEMA}.{config['stream_name']}"
session_source = StreamSource(full_name=STREAM_SOURCE)

# Streaming: what has this shopper looked at in the last 30 minutes?
session_recent_products = Feature(
    name="test_session_recent_products",
    source=session_source,
    entity=["value.user_session"],
    timeseries_column="value.event_time",
    function=AggregationFunction(
        operator=LastDistinct(input="value.product_id", n=10),
        time_window=RollingWindow(window_duration=timedelta(minutes=30)),
    ),
)

# Validate with create_training_set (compute_features doesn't support StreamSource windowed features)
ingestion_df = spark.table(f"{STREAM_SOURCE}_ingestion")
labeled_df = (
    ingestion_df
    .select("value.user_session", "value.event_time")
    .withColumn("label", F.lit(1))
    .limit(100)
)

training_set = fe.create_training_set(
    df=labeled_df,
    features=[session_recent_products],
    label="label",
)
display(training_set.load_df())

In [0]:
backfill = spark.table(f"{CATALOG}.{SCHEMA}.{config['stream_backfill_table']}")
display(backfill)

In [0]:
# Parse the value struct fields into individual columns
parsed_df = backfill.select(
    "value.event_time",
    "value.event_type",
    "value.product_id",
    "value.category_id",
    "value.category_code",
    "value.brand",
    "value.price",
    "value.user_id",
    "value.user_session"
)
display(parsed_df)

In [0]:
# Stream: see config.yml for catalog.schema.stream_name

In [0]:
# Stream: see config.yml for catalog.schema.stream_name

ingestion = spark.table(f"{CATALOG}.{SCHEMA}.{config['stream_backfill_table']}")
display(ingestion)

In [0]:
# Expand the struct fields in 'value' into individual columns
flattened_df = ingestion.select(
    "value.event_time",
    "value.event_type",
    "value.product_id",
    "value.category_id",
    "value.category_code",
    "value.brand",
    "value.price",
    "value.user_id",
    "value.user_session",
    "record_source",
    "stream_record_timestamp",

)
display(flattened_df)